<a href="https://colab.research.google.com/github/elhamod/IS883_Fall_2026/blob/main/Session%2002/IS883_Session02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

From N-grams to Transformers

Use Google Colab for this notebook. **Runtime → Change runtime type → T4 GPU** is recommended but not required; everything below also runs on CPU.

### Where we left off, and where we are going

Last week we built n-gram language models on four public-domain children's books, and we found their ceiling. An n-gram can only look back a fixed number of words. Give it more context and it stops generalizing and starts reciting the book back to you.

This week we keep **exactly the same corpus** and change the model. You will:

1. Rebuild the Week 1 n-gram as our **baseline**.
2. Train a **GPT-2 transformer from scratch** on those same books, and compare.
3. Load the **pretrained GPT-2** that OpenAI trained on 40GB of internet text, and compare again.
4. Score all three fairly with **perplexity** — which turns out to be harder than it sounds.
5. Use HuggingFace for two other jobs entirely: **translation** and **text classification**.

The punchline is worth stating up front, because it is the main idea of the course: the thing that separates a toy language model from a useful one is not mainly the architecture. It is the **amount of text it was trained on**. You will see that measured, not asserted.

# Part 0: Setup

In [ ]:
# Install the libraries this notebook uses (HuggingFace transformers and NLTK).
!pip install transformers nltk --quiet

In [ ]:
# Your BUID seeds every random step so your results are reproducible.
BUID = 123456  # enter ONLY the numerical part

In [ ]:
# Seed every random number generator, and use the GPU if Colab gave us one.
import random
import torch

random.seed(BUID)
torch.manual_seed(BUID)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
if device == "cpu":
    print("TIP: Runtime -> Change runtime type -> T4 GPU will make Part 3 much faster.")


# Part 1: N-gram Baseline

Same four children's books, same lower-casing, same 90/10 split, same seed. Nothing here is new — we are just rebuilding last week's starting point so the comparison is honest.

In [ ]:
# Rebuild Week 1's data: load the four books, lower-case them, and make a 90/10 split.
import nltk
nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.corpus import gutenberg

books = [
    'carroll-alice.txt',          # Alice's Adventures in Wonderland
    'bryant-stories.txt',         # Stories to Tell to Children
    'burgess-busterbrown.txt',    # The Adventures of Buster Bear
    'edgeworth-parents.txt',      # The Parent's Assistant
]

sentences = []
for book in books:
    sentences = sentences + [[word.lower() for word in s] for s in gutenberg.sents(book)]

random.Random(BUID).shuffle(sentences)
split_point = int(0.9 * len(sentences))
train_sentences = sentences[:split_point]
test_sentences  = sentences[split_point:]

print("training sentences:", len(train_sentences))
print("test sentences    :", len(test_sentences))
print("total words       :", sum(len(s) for s in sentences))

Our three models will need the text in two different shapes. The n-gram wants a **list of words**. The transformers want a **plain string**. `TreebankWordDetokenizer` glues the words back into a normal-looking sentence (it knows not to put a space before a comma).

In [ ]:
# The n-gram wants lists of words; the transformers want plain strings.
# TreebankWordDetokenizer glues words back into normal sentences (no space before commas).
from nltk.tokenize.treebank import TreebankWordDetokenizer

detokenizer = TreebankWordDetokenizer()

train_texts = [detokenizer.detokenize(s) for s in train_sentences]
test_texts  = [detokenizer.detokenize(s) for s in test_sentences]

### The baseline: n-gram with `n = 2`

In [ ]:
# Rebuild the Week 1 n-gram (a bigram) as our baseline.
from nltk.lm import Lidstone
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends

def train_ngram_model(n, sentences):
    train_data, vocabulary = padded_everygram_pipeline(n, sentences)
    model = Lidstone(0.01, n)
    model.fit(train_data, vocabulary)
    return model

ngram_model = train_ngram_model(2, train_sentences)

print("n =", ngram_model.order)
print("vocabulary size:", len(ngram_model.vocab), "distinct words")

# Part 2: How a Transformer Reads Text

Before we train anything, one idea has to change: **what counts as a "word."**

GPT-2 does not use whole words. It uses **sub-word pieces** (called BPE tokens). Its vocabulary is 50,257 fixed pieces, and *any* string of English can be spelled out of them.

In [ ]:
# Show how GPT-2 splits text into sub-word (BPE) pieces instead of whole words.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 vocabulary size:", len(tokenizer))
print()

for sentence in ["once upon a time there was a little boy.",
                 "the big data analyst optimized the blockchain.",
                 "redistribution of wealth is a necessity"]:
    pieces = tokenizer.tokenize(sentence)
    print(f'"{sentence}"')
    print("  ->", pieces)
    print(f"({len(pieces)} tokens)")
    print()

The `Ġ` character just means "there was a space before this piece."


# Part 3: Training GPT-2 From Scratch

Now the real experiment. We build a GPT-2 **with random parameters** — the architecture with nothing learned in it — and train it on the training data from the children's books.

This is a small model by modern standards. Real GPT-2 has 12 layers and 768 dimensions; ours has 4 and 256. That is deliberate: it has to fit in Google Colab's memory and finish training during class.

In [ ]:
# Build a small GPT-2 with random (untrained) parameters so it fits and trains in class.
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=128,   # how many tokens of context the model can see
    n_embd=256,        # size of the internal representation
    n_layer=4,         # number of transformer layers
    n_head=4,          # number of attention heads per layer
)

scratch_model = GPT2LMHeadModel(config).to(device)

print("parameters:", f"{sum(p.numel() for p in scratch_model.parameters()):,}")

### Preparing the data

A transformer does not train on one sentence at a time. We glue the whole corpus into one long stream of token IDs, put an end-of-text marker between sentences, and cut the stream into fixed-length blocks.

In [ ]:
# Glue the whole training corpus into one token stream and cut it into fixed-length blocks.
BLOCK_SIZE = 128

all_token_ids = []
for text in train_texts:
    all_token_ids = all_token_ids + tokenizer(text)["input_ids"] + [tokenizer.eos_token_id]

blocks = [all_token_ids[i:i + BLOCK_SIZE]
          for i in range(0, len(all_token_ids) - BLOCK_SIZE, BLOCK_SIZE)]

print("total tokens:", f"{len(all_token_ids):,}")
print("blocks of", BLOCK_SIZE, "tokens:", f"{len(blocks):,}")

### Training

How long this takes depends on your runtime, so we scale the job to the hardware. On a GPU we use the whole corpus; on CPU we use a slice of it so the cell finishes in a few minutes.

In [ ]:
# Scale the training job to the hardware: full corpus on GPU, a slice of it on CPU.
from torch.utils.data import DataLoader

if device == "cuda":
    NUM_EPOCHS, MAX_BLOCKS, BATCH_SIZE = 4, len(blocks), 16
else:
    NUM_EPOCHS, MAX_BLOCKS, BATCH_SIZE = 2, 1000, 8

training_blocks = torch.tensor(blocks[:MAX_BLOCKS])
loader = DataLoader(training_blocks, batch_size=BATCH_SIZE, shuffle=True)

print(f"training on {len(training_blocks):,} blocks for {NUM_EPOCHS} epochs "
      f"= {len(loader) * NUM_EPOCHS} steps")

In [ ]:
# Train the model: for each batch, predict every next token and nudge the weights to do better.
optimizer = torch.optim.AdamW(scratch_model.parameters(), lr=5e-4)

scratch_model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = []
    for step, batch in enumerate(loader):
        batch = batch.to(device)

        loss = scratch_model(input_ids=batch, labels=batch).loss   # predict every next token
        loss.backward()                                            # how should the weights change?
        optimizer.step()                                           # change them
        optimizer.zero_grad()

        running_loss.append(loss.item())
        if step % 25 == 0:
            print(f"  epoch {epoch + 1}, step {step:4d}/{len(loader)}, loss = {loss.item():.3f}")

    print(f"EPOCH {epoch + 1} average loss = {sum(running_loss) / len(running_loss):.3f}")

The loss should fall steadily. It is the model's average surprise per token — the same quantity perplexity is built from, just on a log scale.

### What did it learn?

In [ ]:
# Generate a continuation of a prefix using a HuggingFace language model.
def generate_with_transformer(model, prefix, max_new_tokens=40, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()

    inputs = tokenizer(prefix, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,   # how many tokens to generate
            do_sample=True,                  # sample the next token, like we did with the n-gram
            top_k=50,                        # only consider the 50 most likely next tokens
            temperature=0.9,                 # higher = more random, lower = more predictable
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


for i in range(3):
    print(generate_with_transformer(scratch_model, "once upon a time"))
    print("---")

**Experiment.** Two knobs control how adventurous the generation is: `top_k` (how many candidate next tokens to consider) and `temperature` (higher = more random). Change them in the cell above and re-run — try `temperature=0.2` versus `temperature=1.5`. Add `seed=BUID + i` to the call if you want repeatable output.

**Question 1.** Look at that output honestly. Is it better or worse than the Week 1 n-gram? Why might that be, given that a transformer is a far more powerful model?

**Answer**

*Provide your answer here*

# Part 4: The Pretrained GPT-2

Same architecture family, same tokenizer, one difference: these weights were trained by OpenAI on roughly 40GB of internet text — something like **8 billion words**, versus our 320,000.

That is a factor of about 25,000.

In [ ]:
# Load OpenAI's pretrained GPT-2 (trained on ~8 billion words) and generate from it.
pretrained_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

print("parameters:", f"{sum(p.numel() for p in pretrained_model.parameters()):,}")

for i in range(3):
    print(generate_with_transformer(pretrained_model, "once upon a time", seed=BUID + i))
    print("---")

### Three models, one prefix

The fairest qualitative comparison: give all three the same prompts and read the results side by side.

In [ ]:
# Week 1's n-gram generator, reproduced here so we can compare all three models side by side.
def generate_with_ngram(model, prefix, num_words=40, seed=None):
    n = model.order
    rng = random.Random(seed) if seed is not None else None
    prefix_words = nltk.word_tokenize(prefix.lower())
    context = ["<s>"] * (n - 1) + prefix_words
    generated = list(prefix_words)
    for _ in range(num_words):
        next_word = model.generate(text_seed=context[-(n - 1):] if n > 1 else [],
                                   random_seed=rng)
        if next_word == "</s>":
            break
        if next_word == "<s>":
            continue
        generated.append(next_word)
        context.append(next_word)
    return " ".join(generated)


prompts = [
    "once upon a time",
    "the little girl said",
    "the quarterly earnings report",   # not children's-book language at all
]

for prompt in prompts:
    print("=" * 70)
    print(f'PROMPT: "{prompt}"')
    print("=" * 70)
    print("N-GRAM (Week 1)      :", generate_with_ngram(ngram_model, prompt, 35, seed=BUID))
    print()
    print("GPT-2 FROM SCRATCH   :", generate_with_transformer(scratch_model, prompt, 35, seed=BUID))
    print()
    print("GPT-2 PRETRAINED     :", generate_with_transformer(pretrained_model, prompt, 35, seed=BUID))
    print()

**Question 2.** Compare the three on the prompt `"the quarterly earnings report"`. What does each model do, and what does that tell you about what each one actually learned?

**Answer**

*Provide your answer here*

# Part 5: Comparing Them Fairly with Perplexity

Now the numbers. And here there is a trap that catches professionals, so it is worth slowing down.

Perplexity is "average surprise per **step**." But the three models do not take the same steps. The n-gram predicts one **word** at a time. GPT-2 predicts one **BPE token** at a time, and a sentence is roughly 1.3 tokens per word. Comparing the raw perplexities is like comparing a car's fuel use in miles-per-gallon against another's in litres-per-100km: same idea, different units, and the smaller number is not automatically the better car.

The fix is to put both on a **per-word** basis. We add up each model's total surprise over the exact same test sentences, then divide by the same word count for everyone.

In [ ]:
# Perplexity helpers: the n-gram is scored per word, the transformers per BPE token.
# per_word() puts both on the same per-word footing so the comparison is fair.
import math
from nltk.util import ngrams

# One shared evaluation set, so every model is graded on identical sentences.
eval_sentences = test_sentences
eval_word_count = sum(len(s) for s in eval_sentences)
print("evaluating on", len(eval_sentences), "sentences /", eval_word_count, "words")


def ngram_perplexity(model, sentences):
    """Week 1's perplexity, unchanged. Also reports how many predictions it made."""
    n = model.order
    all_ngrams = []
    for sentence in sentences:
        tokens = model.vocab.lookup(list(pad_both_ends(sentence, n)))
        all_ngrams = all_ngrams + list(ngrams(tokens, n))
    return model.perplexity(all_ngrams), len(all_ngrams)


def transformer_perplexity(model, sentences):
    """Perplexity of a HuggingFace model, per BPE token. Also reports the token count.

    Anything longer than the model's context window is read in consecutive
    chunks that fit, so no text is skipped and nothing has to be hardcoded.
    """
    model.eval()
    window = model.config.n_positions          # how much this model can read at once
    total_surprise = 0
    total_tokens = 0
    for sentence in sentences:
        ids = tokenizer(detokenizer.detokenize(sentence), return_tensors="pt").input_ids.to(device)

        for start in range(0, ids.shape[1], window):
            chunk = ids[:, start:start + window]
            if chunk.shape[1] < 2:                          # a lone token has nothing to predict
                continue
            with torch.no_grad():
                average_surprise = model(input_ids=chunk, labels=chunk).loss.item()
            predictions = chunk.shape[1] - 1                # every token except the first
            total_surprise = total_surprise + average_surprise * predictions
            total_tokens = total_tokens + predictions

    return math.exp(total_surprise / total_tokens), total_tokens


def per_word(perplexity, predictions, words):
    """Convert a per-prediction perplexity into a per-word one.

    perplexity = exp(total_surprise / predictions), so raising it to
    (predictions / words) gives exp(total_surprise / words): the surprise per word.
    """
    return perplexity ** (predictions / words)

In [ ]:
# Score all three models on the shared test set, converted to a per-word basis so the
# comparison is fair (the n-gram predicts words, the transformers predict BPE tokens).
ngram_ppl, ngram_predictions = ngram_perplexity(ngram_model, eval_sentences)
scratch_ppl, scratch_predictions = transformer_perplexity(scratch_model, eval_sentences)
pretrained_ppl, pretrained_predictions = transformer_perplexity(pretrained_model, eval_sentences)

ngram_word_ppl = per_word(ngram_ppl, ngram_predictions, eval_word_count)
scratch_word_ppl = per_word(scratch_ppl, scratch_predictions, eval_word_count)
pretrained_word_ppl = per_word(pretrained_ppl, pretrained_predictions, eval_word_count)

print(f"{'model':<24} {'per word perplexity':>20}")
print("-" * 46)
print(f"{'n-gram (bigram)':<24} {ngram_word_ppl:>20.1f}")
print(f"{'GPT-2 from scratch':<24} {scratch_word_ppl:>20.1f}")
print(f"{'GPT-2 pretrained':<24} {pretrained_word_ppl:>20.1f}")
print()

**Question 3.** Rank the three models. Does the ranking match what you judged by eye in Part 4? And what is the single biggest factor explaining the gap?

**Answer**

*Provide your answer here*

# Part 6: HuggingFace Beyond Text Generation

Everything so far has been one task: predict the next word. But the same library gives you thousands of models fine-tuned for entirely different jobs, and `pipeline` hides essentially all of the machinery.

The pattern is always the same three lines: pick a task, pick a model, call it.

## 6.1 Translation: English → Spanish

In [ ]:
# Task 1 for the pretrained library: translate English to Spanish with a ready-made model.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-es"
translation_tokenizer = AutoTokenizer.from_pretrained(model_name)
translator = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

to_translate = [
    "Once upon a time there was a little girl who lived near the forest.",
    "The quarterly earnings report exceeded analyst expectations.",
    "Please send me the invoice before Friday afternoon.",
]

for sentence in to_translate:
    inputs = translation_tokenizer(sentence, return_tensors="pt").to(device)   # text -> token ids
    output_ids = translator.generate(**inputs)                                 # ids  -> ids
    result = translation_tokenizer.decode(output_ids[0], skip_special_tokens=True)   # ids -> text

    print("EN:", sentence)
    print("ES:", result)
    print("---")

In [ ]:
# Compare model sizes: the translation model vs. the pretrained GPT-2.
print("translation model parameters:",
      f"{sum(p.numel() for p in translator.model.parameters()):,}")
print("pretrained GPT-2 parameters :",
      f"{sum(p.numel() for p in pretrained_model.parameters()):,}")

## 6.2 Text classification: sentiment

Classification is the workhorse task of applied NLP: route this ticket, flag this review, tag this email. Start with the default sentiment model on a few reviews we write ourselves.

In [ ]:
# Task 2: sentiment classification with a ready-made pipeline on a few reviews we write.
from transformers import pipeline

sentiment = pipeline("sentiment-analysis",
                     model="distilbert-base-uncased-finetuned-sst-2-english",
                     device=0 if device == "cuda" else -1)

reviews = [
    "The food was incredible and the staff could not have been friendlier.",
    "Waited 45 minutes for a cold burger. Never coming back.",
    "It was fine. Nothing special, nothing terrible.",
    "Overpriced, but I have to admit the dessert was worth it.",
]

for review in reviews:
    result = sentiment(review)[0]
    print(f"{result['label']:<9} ({result['score']:.3f})  {review}")

**Question 4.** Look at the third and fourth reviews in the output above — one deliberately neutral, one mixed. What label did each get, and with what confidence? What does that tell you about what this model can and cannot express?


**Answer**

*Provide your answer here*

### A finer-grained classifier

POSITIVE/NEGATIVE is coarse. We switch to a model that predicts **1 to 5 stars**, so it can express *how* positive or negative each review is.


In [ ]:
# Switch to a model that predicts 1-5 stars instead of just positive/negative.
star_classifier = pipeline("sentiment-analysis",
                           model="nlptown/bert-base-multilingual-uncased-sentiment",
                           device=0 if device == "cuda" else -1)

results = star_classifier(reviews)

for review, result in zip(reviews, results):
    print(f"{result['label']:<9} ({result['score']:.3f})  {review}")

In [ ]:
# Run the same reviews through the star classifier a second time and check the labels match.
second_run = star_classifier(reviews)
print("Identical labels on a re-run:",
      [r['label'] for r in results] == [r['label'] for r in second_run])

**Question 5.** The classifier gave the same labels twice. Now open Claude in two fresh chats, paste the same four reviews into each, and ask for a 1-to-5-star rating for every review. Do the two chats agree with each other? Which matters more for a business that has to explain a decision to an auditor: that the model is smart, or that it is *reproducible*?

**Answer**

*Provide your answer here*

# Part 7: Follow up Questions and Exercises

**Exercise 1.** Train the from-scratch GPT-2 for more epochs (or switch to a GPU runtime and rerun Part 3). Does its per-word perplexity drop below the n-gram's? Roughly how much training does it take to catch up?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*